In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from scipy.signal import max_len_seq


In [ ]:

# ==========================================
# 1. 实验环境参数与硬件宏定义
# ==========================================
# 距离参数 (单位: mm)
Z1_SOURCE_TO_SLM = 300.0   # 光源到 SLM 的距离
Z2_SLM_TO_SENSOR = 15.0    # SLM 到 传感器的距离

# 硬件固定参数
SLM_RES = (768, 1024)      # SLM 分辨率 (高, 宽)
SLM_PITCH_MM = 0.036       # SLM 像素间距

# 传感器参数 (Sony IMX183)
SENSOR_WIDTH_MM = 13.133
SENSOR_HEIGHT_MM = 8.755

# 算法控制参数
SAFE_MARGIN = 0.7         # 安全系数，控制影子占传感器的比例
OUTPUT_DIR = "./msl_patterns_cv2"
INTERPOLATION_METHOD = cv2.INTER_NEAREST  # 最近邻插值，确保 M 序列保持二值特性

In [ ]:

# ==========================================
# 2. 核心计算与生成函数
# ==========================================
def generate_mask_cv2(n_bits):
    """
    基于 OpenCV 实现的 M 序列掩模生成。
    使用浮点缩放替代整数 repeat，确保不同 N 值在 Margin 变化时同步缩放。
    """
    # 几何放大率计算
    magnification = (Z1_SOURCE_TO_SLM + Z2_SLM_TO_SENSOR) / Z1_SOURCE_TO_SLM
    sensor_limit_mm = min(SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM)
    
    # 计算 SLM 上允许的总像素宽度 (不受 N 值影响的统一标准)
    allowed_slm_size_mm = (sensor_limit_mm * SAFE_MARGIN) / magnification
    target_pixels = int(allowed_slm_size_mm / SLM_PITCH_MM) 

    # 生成 M 序列原始二值矩阵
    try:
        m_seq = max_len_seq(n_bits)[0]
    except:
        m_seq = max_len_seq(n_bits, taps=None)[0]
        
    mask_logic = np.outer(m_seq * 2 - 1, m_seq * 2 - 1)
    mask_binary = np.where(mask_logic > 0, 1, 0).astype(np.uint8)

    # 使用 OpenCV 强制缩放到目标像素尺寸，消除整数阶梯误差
    mask_scaled = cv2.resize(mask_binary, (target_pixels, target_pixels), 
                             interpolation=INTERPOLATION_METHOD)

    # 居中填入全屏画布
    full_canvas = np.zeros(SLM_RES, dtype=np.uint8)
    h, w = mask_scaled.shape
    y_start = (SLM_RES[0] - h) // 2
    x_start = (SLM_RES[1] - w) // 2
    full_canvas[y_start:y_start+h, x_start:x_start+w] = mask_scaled * 255
    
    return full_canvas

# ==========================================
# 3. 自动化任务流 (单张 + 混合拼接)
# ==========================================
def run_pipeline():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    # 缓存各层图像
    results = {}
    n_values = [3, 4, 5, 6]

    print(f"开始任务：Margin = {SAFE_MARGIN}")
    
    # A. 生成四张独立的完整图
    for n in n_values:
        img = generate_mask_cv2(n)
        results[n] = img
        
        save_path = os.path.join(OUTPUT_DIR, f"Full_N{n}.bmp")
        Image.fromarray(img, mode='L').save(save_path)
        print(f"  - 已保存单张: N={n}")

    # B. 执行拼接逻辑 (象限对调：TL:N6, TR:N4, BL:N5, BR:N3)
    final_composite = np.zeros(SLM_RES, dtype=np.uint8)
    cy, cx = SLM_RES[0] // 2, SLM_RES[1] // 2

    # 分象限提取并组合
    final_composite[0:cy, 0:cx] = results[6][0:cy, 0:cx] # 左上 <- N6
    final_composite[0:cy, cx:]  = results[4][0:cy, cx:] # 右上 <- N4
    final_composite[cy:, 0:cx]  = results[5][cy:, 0:cx] # 左下 <- N5
    final_composite[cy:, cx:]   = results[3][cy:, cx:]  # 右下 <- N3

    # 保存拼接结果
    comp_path = os.path.join(OUTPUT_DIR, f"Composite_N6453_M{int(SAFE_MARGIN*100)}.bmp")
    Image.fromarray(final_composite, mode='L').save(comp_path)
    print(f"✅ 混合拼接图案已完成并保存至: {comp_path}")

    # C. 纯净预览
    plt.figure(figsize=(10, 8))
    plt.imshow(final_composite, cmap='gray')
    plt.title(f"CV2 Linear Scaled Composite (Margin={SAFE_MARGIN})")
    plt.axis('off')
    plt.show()

    return results

# 启动程序
if __name__ == "__main__":
    results = run_pipeline()

In [ ]:
# ==========================================
# 4. 后处理：利用全局 results 保存四个独立的遮盖象限图
# ==========================================

# 重新获取全局中心点
_cy, _cx = SLM_RES[0] // 2, SLM_RES[1] // 2

# 定义操作逻辑映射 (TL:N6, TR:N4, BL:N5, BR:N3)
quadrant_ops = {
    6: ("Top-Left",     (0, _cy, 0, _cx)),
    4: ("Top-Right",    (0, _cy, _cx, SLM_RES[1])),
    5: ("Bottom-Left",  (_cy, SLM_RES[0], 0, _cx)),
    3: ("Bottom-Right", (_cy, SLM_RES[0], _cx, SLM_RES[1]))
}

print(f"开始后处理任务：从全局变量读取图像并应用遮盖...")

for n, (name, (y_start, y_end, x_start, x_end)) in quadrant_ops.items():
    # 直接从已存在的全局 results 字典中读取图像副本
    full_img = results[n].copy()
    
    # 创建全黑画布
    masked_img = np.zeros_like(full_img)
    
    # 只将指定象限的内容复制到黑色画布上
    # 这样就实现了：保留该象限，其余部分（包括其他象限和背景边缘）全部遮盖
    masked_img[y_start:y_end, x_start:x_end] = full_img[y_start:y_end, x_start:x_end]
    
    # 保存结果
    masked_filename = f"Masked_Only_N{n}_{name}.bmp"
    masked_save_path = os.path.join(OUTPUT_DIR, masked_filename)
    Image.fromarray(masked_img, mode='L').save(masked_save_path)
    
    print(f"  - 已生成并保存遮盖图: {masked_filename}")

print(f"\n所有象限遮盖图已保存至目录: {OUTPUT_DIR}")